# FlowPolicy Lowdim - Franka Kitchen experiment (Google Colab)

Notebook ini hanya **memanggil** kode yang sudah ada di repo (`FlowPolicy/` dan `scripts/`).
Tidak ada logika model yang ditulis ulang di sini. Pastikan runtime Colab pakai **GPU** (Runtime -> Change runtime type -> GPU).

## 1. Cek GPU dan versi Python

In [ ]:
!nvidia-smi || echo 'No GPU'
!python --version
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Ambil repo FlowPolicy

Pilih salah satu:
* (A) clone repo publik dari GitHub, lalu copy file baru (encoder lowdim, dataset/runner kitchen, scripts) yang Anda buat di repo lokal ke Colab via `files.upload()` atau Google Drive.
* (B) langsung upload seluruh folder repo (ZIP) yang Anda miliki (sudah berisi semua file baru) ke Colab dan extract.

Cell di bawah pakai opsi (B): extract ZIP yang Anda upload.

In [ ]:
# OPSI A: clone dari GitHub publik. Aktifkan jika repo Anda sudah dipublish.
# !git clone https://github.com/zql-kk/FlowPolicy.git
# %cd FlowPolicy

# OPSI B: upload ZIP repo Anda lalu extract
from google.colab import files
uploaded = files.upload()  # pilih file FlowPolicy.zip
import zipfile, os
for name in uploaded:
    with zipfile.ZipFile(name) as z:
        z.extractall('/content/')
    print('extracted', name)
%cd /content/FlowPolicy
!ls

## 3. Install dependensi yang belum ada di Colab

Stack Colab modern sudah punya `torch`, `omegaconf`, `dill`, `einops`, `wandb`, `numba`, `diffusers`, `gymnasium`, `scikit-learn`, `imageio`, `matplotlib`. Kita install sisanya.

In [ ]:
!pip install -q hydra-core==1.2.0 zarr==2.12.0 \
    'minari[hdf5]' gymnasium-robotics mujoco \
    termcolor

## 4. Install paket FlowPolicy (editable)

`setup.py` ada di `FlowPolicy/` (subfolder paket Python). `find_packages()` otomatis termasuk modul baru.

In [ ]:
!pip install -q -e FlowPolicy/
import importlib
import flow_policy_3d
importlib.reload(flow_policy_3d)
print('flow_policy_3d ok')

## 5. (Opsional) Login Weights & Biases

Skrip orkestrasi tidak wajib pakai wandb (training loop kustom hanya menulis JSON). Set offline saja agar tidak crash.

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'

## 6. Pra-unduh dataset Minari `D4RL/kitchen/complete-v2`

Sekali ini saja. Hasilnya juga di-cache ke `data/kitchen_complete_v2_episodes.npz` oleh `KitchenLowdimDataset` agar pemuatan berikutnya tidak perlu `minari` lagi.

In [ ]:
!python -c "import minari; minari.download_dataset('D4RL/kitchen/complete-v2'); print('downloaded')"

## 7. Smoke test (cepat) - pastikan pipeline jalan sebelum eksekusi penuh

n_iter=2, cv=2, proxy_epochs=1, hanya 1 seed.

In [ ]:
!python scripts/run_kitchen_experiment.py \
    --seeds 0 \
    --n_iter 2 --cv 2 \
    --proxy_epochs 1 --proxy_steps_per_epoch 5 \
    --rollout_every 10 \
    --inference_episodes 3 \
    --eval_episodes_during_training 2 \
    --output_root results_kitchen_smoke

## 8. Eksekusi penuh: 3 seed x 2 tipe x random search n_iter=100 cv=5

Ini panjang (jam-an di T4/A100). Sesuaikan `--proxy_epochs` dan `--proxy_steps_per_epoch` jika ingin lebih cepat. `--n_iter` dan `--cv` mengikuti spek user.

Subset tipe (opsional): `--types no_preprocess with_preprocess` (default keduanya). Untuk satu tipe saja mis. `--types with_preprocess`.

In [ ]:
!python scripts/run_kitchen_experiment.py \
    --seeds 0 42 101 \
    --types no_preprocess with_preprocess \
    --n_iter 100 --cv 5 \
    --proxy_epochs 5 --proxy_steps_per_epoch 50 \
    --rollout_every 100 \
    --inference_episodes 20 \
    --eval_episodes_during_training 5 \
    --output_root results_kitchen

## 9. Tampilkan plot success rate

In [ ]:
from IPython.display import Image
Image('results_kitchen/success_rate.png')

## 10. Cetak ringkasan semua model

In [ ]:
import json
with open('results_kitchen/all_summaries.json') as f:
    summaries = json.load(f)
for s in sorted(summaries, key=lambda x: -x['best_success_rate']):
    print(f"seed={s['seed']:>3} type={s['type']:<14} SR={s['best_success_rate']:.4f}")

## 11. Tampilkan ringkasan inferensi pemenang + video

In [ ]:
import json
with open('results_kitchen/inference_winner/inference_summary.json') as f:
    inf = json.load(f)
print(json.dumps({k: inf[k] for k in ['mean_success_rate', 'mean_full_success', 'mean_latency_ms', 'n_episodes'] if k in inf}, indent=2))

In [ ]:
from IPython.display import Video
Video('results_kitchen/inference_winner/best_model_inference.mp4', embed=True, width=480)

## 12. (Opsional) Simpan hasil ke Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r results_kitchen /content/drive/MyDrive/flowpolicy_kitchen_results